In [3]:
from datasets import load_dataset
from collections import Counter
import re

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-v1"
)

print(dataset)

# Use training data
text = "\n".join(dataset["train"]["text"])

print("\nCorpus loaded successfully!")
print("Total characters:", len(text))

text = text.lower()

# Keep only alphabetic words
words = re.findall(r'\b[a-z]+\b', text)

print("Total words:", len(words))
print("First 20 words:", words[:20])

unigram = Counter(words)

print("\nTop 10 Unigram Words:")
print(unigram.most_common(10))

bigram = Counter()

for i in range(len(words) - 1):

    pair = (words[i], words[i + 1])

    bigram[pair] += 1

print("\nTop 10 Bigrams:")
print(bigram.most_common(10))

trigram = Counter()

for i in range(len(words) - 2):

    triple = (
        words[i],
        words[i + 1],
        words[i + 2]
    )

    trigram[triple] += 1

print("\nTop 10 Trigrams:")
print(trigram.most_common(10))

# Unigram probability
def unigram_probability(word):

    return unigram[word] / len(words)


# Bigram probability
def bigram_probability(word1, word2):

    count_bigram = bigram[(word1, word2)]

    count_word = unigram[word1]

    if count_word == 0:
        return 0

    return count_bigram / count_word


# Trigram probability
def trigram_probability(word1, word2, word3):

    count_trigram = trigram[
        (word1, word2, word3)
    ]

    count_bigram = bigram[
        (word1, word2)
    ]

    if count_bigram == 0:
        return 0

    return count_trigram / count_bigram

def predict_next_words(sentence, top_n=5):

    # Tokenize input sentence
    input_words = re.findall(
        r'\b[a-z]+\b',
        sentence.lower()
    )

    if len(input_words) == 0:
        return []

    predictions = {}

    if len(input_words) >= 2:

        word1 = input_words[-2]
        word2 = input_words[-1]

        for (w1, w2, w3), count in trigram.items():

            if w1 == word1 and w2 == word2:

                probability = (
                    count / bigram[(word1, word2)]
                )

                predictions[w3] = probability


    if not predictions:

        last_word = input_words[-1]

        for (w1, w2), count in bigram.items():

            if w1 == last_word:

                probability = (
                    count / unigram[last_word]
                )

                predictions[w2] = probability

    if not predictions:

        for word, count in unigram.most_common():

            predictions[word] = (
                count / len(words)
            )

            if len(predictions) >= top_n:
                break

    predictions = sorted(
        predictions.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]

query = input(
    "\nEnter a sentence or partial sentence: "
)

top_n = int(
    input("Enter number of predictions (3-5): ")
)


predictions = predict_next_words(
    query,
    top_n
)


print("\n================================")
print("SMART NEXT-WORD PREDICTOR")
print("================================")

print("\nOriginal Input:")
print(query)

print("\nTop Next-Word Predictions:")

if predictions:

    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):

        print(
            f"{i}. {word} - {probability:.3f}"
        )

else:

    print("No predictions found.")

print("\n================================")
print("TESTING MULTIPLE SENTENCES")
print("================================")

test_sentences = [
    "machine learning",
    "the united",
    "artificial intelligence",
    "new york",
    "computer science"
]

for sentence in test_sentences:

    predictions = predict_next_words(
        sentence,
        3
    )

    print("\nInput:", sentence)

    if predictions:

        for word, probability in predictions:

            print(
                f"{word} - {probability:.3f}"
            )

    else:

        print("No prediction found.")

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Corpus loaded successfully!
Total characters: 10791252
Total words: 1683604
First 20 words: ['valkyria', 'chronicles', 'iii', 'no', 'valkyria', 'unk', 'chronicles', 'japanese', 'lit', 'valkyria', 'of', 'the', 'battlefield', 'commonly', 'referred', 'to', 'as', 'valkyria', 'chronicles', 'iii']

Top 10 Unigram Words:
[('the', 130768), ('of', 57030), ('unk', 54625), ('and', 50735), ('in', 45015), ('to', 39521), ('a', 36523), ('was', 21008), ('on', 15140), ('as', 15058)]

Top 10 Bigrams:
[(('of', 'the'), 17480), (('in', 'the'), 12778), (('unk', 'unk'), 6256), (('to', 'the'), 6084), (('the', 'unk'), 4632), (('on', 'the'), 4523), (('and', 'the'), 4472), (('unk', 'and'), 4368), (('for', 'the'), 3741), (('at', 'the'), 3242)]

Top 10 Trigra


Enter a sentence or partial sentence:  machine learning
Enter number of predictions (3-5):  5



SMART NEXT-WORD PREDICTOR

Original Input:
machine learning

Top Next-Word Predictions:
1. that - 0.133
2. curve - 0.093
3. the - 0.080
4. to - 0.080
5. about - 0.067

TESTING MULTIPLE SENTENCES

Input: machine learning
that - 0.133
curve - 0.093
the - 0.080

Input: the united
states - 0.764
kingdom - 0.178
nations - 0.043

Input: artificial intelligence
ai - 0.154
was - 0.077
research - 0.077

Input: new york
times - 0.164
city - 0.146
state - 0.060

Input: computer science
health - 0.071
with - 0.071
engineering - 0.071
